# Module 6: Running it for real

You run the regression set against the agent with every fix in place. It asks one question for each failure you fixed in modules 1 to 5, and checks what the agent did each time. You read each verdict and its reason, then take away a checklist for the field: when a customer says their agent misbehaves, it tells you which layer to look at first. The module takes about 10 minutes.

In [ ]:
# Load the lab's notebook helpers, and work in the agent's folder.
import labnb as lab
%cd ../agent

## Start from module 6

This cell puts the agent where module 6 starts, with every fix from modules 2 to 5 in place. It
copies any edits you made to `agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
# Reset the agent to module 6's starting state: every fix from modules 2 to 5.
!python reset.py 6

## 1. Run the regression set

The regression set is [`regression.py`](../agent/regression.py), a script that asks the agent one
question for each failure you fixed, then checks each run. It has five cases. Each is named after
one of the lab's questions in [`questions.py`](../agent/questions.py), the names you used with `--q`:

- `m1`, the K-400 coolant pump part number: the tool call fires (modules 1 and 2).
- `m3`, the coolant pump at the Denver depot: the loop uses the tool results and ends on the model's stop (module 3).
- `m4`, the sensor at the Portland depot: the failed stock check is reported (module 4).
- `m5`, fault F-21: the answer comes from a service bulletin and cites it (module 5).
- `m5-miss`, the K-900 service interval: no bulletin covers it, and the search says so (module 5).

```mermaid
flowchart TB
  subgraph Cases["Five cases"]
    direction TB
    C1["m1: the part number"] ~~~ C3["m3: the Denver chain"] ~~~ C4["m4: the Portland depot"]
    C5["m5: fault F-21"] ~~~ C5M["m5-miss: the K-900"]
  end
  Cases --> AG["The agent with every fix"]
  AG --> REC["Run records: tools run, results, how each run ended"]
  REC --> J{"Fixed checks in regression.py"}
  J --> REP["A verdict and reason per case, and a report"]
```

Fixed rules in `regression.py` check each case against the run's record: which tools ran, what
they returned and how the run ended. No second model grades the answer, so the same run always
gets the same verdict. Two cases also look for words in the answer: `m5` must cite `SB-2024-102`,
and `m4` must say the stock couldn't be checked.

The next cell runs all five cases. It takes a few minutes, and nothing prints until the last case
has finished. Wait for it rather than running it again.

In [ ]:
# Run all five cases against the agent and print each verdict with its reason.
# With MLflow on, the run is also recorded there as an evaluation run named closing-run.
lab.regression(name="closing-run")

## 2. Read the verdicts

Each case prints a verdict and the reason for it, and the last line counts them.

- `PASS`: the run did what that module's fix makes it do.
- `FAIL`: it didn't, and the reason says what the check didn't find.
- `SKIP`: the case can't apply, because the agent isn't offered the tool it needs. Every fix is in
  place here, so a SKIP means something is missing. See *If the result differs*.

A run the model cut off is FAIL too. If the model runs out of room before it finishes its answer,
the run ends on `length`, and its case is FAIL whatever else its check found. The reason then reads
"the run was cut off before it finished its answer (the model ran out of room); run the set again".

**Your read:** suppose `m5` came back FAIL with the reason "SB-2024-102 was retrieved but the
answer omitted its source citation". Did the search go wrong, or the answer?

<details><summary>Check your answer</summary>

The answer. The search found the right bulletin, and the model left the citation out of what it
wrote. Look at the instructions module 5 added, which tell the agent to name the bulletins it
retrieved.
</details>

**This is the module's success signal:** all five cases PASS, and the last line reads
`5 passed, 0 failed, 0 skipped`.

## 3. Symptom to layer

The regression run also wrote a report under `agent/runs/`. It repeats the five results, then ends
with the checklist this lab gives you to take away: what a customer might say about their agent,
and the layer to look at first.

**Your read:** before you open the report, think back over modules 2 to 5. A customer says their
agent told them it ordered a part, but no order was ever placed. Which layer do you look at first?

<details><summary>Check your answer</summary>

Tool returns (module 4). It's the "confidently reports work it did not do" row: check whether the
ordering tool's result can say that it failed. In module 4, an unknown depot came back looking like
"no stock" until the tool returned `bad_request`. Here's the whole checklist:

| A customer says... | Look first at | Layer |
|---|---|---|
| The agent ignores its tools | How the model is served (tool-call parser), not the prompt | Model interface (module 2) |
| It loops, or runs are expensive | Whether the loop appends tool results and reads the stop condition | Agent loop (module 3) |
| It confidently reports work it did not do | Whether tool returns can express failure | Tool returns (module 4) |
| It makes things up about their own data | Whether anything grounds it in a source | Knowledge search (module 5) |
| Who can see what? | Whether tool access goes through a governed path on the platform | Governed tool access: the next step after this lab, with no hands-on module |
</details>

The next cell shows the newest report. The checklist at its end is the part to read.

In [ ]:
# Show the newest regression report from agent/runs/.
reports = sorted(lab.RUNS.glob("regression-*.md"))
lab.display(lab.Markdown(reports[-1].read_text() if reports else "No report yet."))

The first four rows each point to a module you worked through. The last row, "Who can see what?",
is about which tools each agent may use. This lab has no hands-on step for it. On a shared
platform, tool access can go through a governed path that sets which tools each agent is offered.
*Where to go next* says where to read about it.

To keep the report, right-click it in the file browser and choose **Download**. The checklist is
also on this module's page in Showroom: open *Check your answer* under *Questions to think about*.

A clean run shows that the agent still handles the failures this lab fixed. It says nothing about
failures nobody has thought of yet. The `m5` check confirms that the answer cites SB-2024-102, but
it can't tell whether what the answer says is right. Checking that takes a person, or a model
grading the answer, and this lab leaves it out.

The report's last lines, under "Covered by discussion", name two more failures this set doesn't
test. The first is the model or the prompt changing underneath an agent that works. Running this
set again after a change like that is how you'd catch it. The second is cost when many people use
the agent at once. These checks ask one question at a time, so they can't show it.

## 4. The evaluation run in MLflow (stretch goal)

Skip this step if you're short of time.

With MLflow on, `name="closing-run"` in step 1 also recorded the run in MLflow as an **evaluation
run**: one row per case, each with its trace and the same verdict and reason the cell printed.
MLflow keeps each verdict on the case's trace as an assessment named `check`. The run also keeps
the counts, `passed`, `failed` and `skipped`, and it's linked to the version of the agent that ran
it. Yours is `kestrel-m6-start`, the name for module 6's starting state. If you'd changed anything
in the agent since the reset, it would read `kestrel-m6-start-modified`.

Open the link the regression cell printed, or the link below, which lists all your evaluation runs.
Select `closing-run` to see its five cases, each with its verdict, reason and trace. On the run's
page, look for the agent version and the three counts, which should match the regression cell's
last line.

In [ ]:
# A link to your evaluation runs in MLflow.
lab.links(("Your evaluation runs in MLflow", lab.experiment_url("evaluation-runs")))

## Where to go next

In this lab the agent's tools are written into its own code, and each one carries its own address
and credential. On a shared platform, governed tool access puts one place in charge of which tools
each agent may see and call, so adding or withdrawing a tool doesn't mean changing every agent.
That's the "Who can see what?" row of the checklist. MCP Gateway, which comes with Red Hat
Connectivity Link, is one way to provide it on OpenShift: agents reach their tools through the
gateway, and the platform sets which tools each one is offered. This lab has no hands-on step for
it, so read its documentation when you're ready to try it.

## If the result differs

Success is 5 PASS, 0 FAIL, 0 SKIP. A cell that finishes without an error doesn't mean its cases
passed, so read each case's verdict and reason.

If a case fails because its run was cut off, the model used up its allowance of 4,096 tokens for
one model call before it finished its answer. The allowance counts the model's reasoning as well as
its answer. Run the set again.

If `m1` or `m3` fails, check which model endpoint the agent calls: add a cell and run
`print(lab.endpoint())`. It should start with `qwen3-8b-b`, the endpoint module 2 moved the agent
to. If it doesn't, run the reset cell again, then the regression cell.

If `m5` or `m5-miss` shows SKIP, the agent wasn't offered `search_bulletins`, module 5's search
tool. Run the reset cell again, then the regression cell.

For any other FAIL, the reason says what the check didn't find. For `m5`, it tells you whether
SB-2024-102 wasn't retrieved at all, or was retrieved but left out of the answer.

If the regression cell says the evaluation wasn't recorded in MLflow, every result is still in the
report and the module is complete. Only the stretch goal needs MLflow.